In [ ]:
# __ROOTBOOT__ ensure project root on sys.path (auto-added; safe to keep)
import os as _os, sys as _sys
_r = _os.path.abspath('')
while _r != _os.path.dirname(_r) and not _os.path.exists(_os.path.join(_r, '.project_root')):
    _r = _os.path.dirname(_r)
if _os.path.exists(_os.path.join(_r, '.project_root')) and _r not in _sys.path:
    _sys.path.insert(0, _r)


# Overnight Premium — Implementation Comparison

Loads standardized signal trades from `results/overnight_premium_trades.csv`
and tests multiple sizing implementations.

**Note (Discussion Point):** Trades are **monthly aggregates** — each row represents
one calendar month's overnight premium return. With ~300 trades, the `vol_targeting`
lookback of 60 consumes 5 years of history. Because the overnight premium is earned
daily but aggregated to monthly here, `vol_targeting` is less informative than it would
be with daily/weekly trades. See `todo/refactor_discussion_points.md` §1.2.

Also, no `stop_price` column exists → `risk_based` uses fallback sizing.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, json

from _shared.implementations import (
    simple_bet, risk_based, vol_targeting, compare_implementations
)

STARTING_CAPITAL = 100_000
STRATEGY_NAME    = "Overnight Premium"
SAVE_NAME        = "overnight_premium"

pd.set_option("display.max_columns", None)

## 2. Load Signal Trades

In [ ]:
trades = pd.read_csv("results/overnight_premium_trades.csv", parse_dates=["entry_time", "exit_time"])
print(f"Loaded {len(trades)} trades")
print(f"Period: {trades['entry_time'].iloc[0].date()} → {trades['exit_time'].iloc[-1].date()}")
print(f"Columns: {list(trades.columns)}")
print(f"Has stop_price: {'stop_price' in trades.columns}")
print(f"Directions: {dict(trades['direction'].value_counts())}")
print(f"Exit reasons: {dict(trades['exit_reason'].value_counts())}")
print()
print("NOTE: Monthly aggregates. Each 'trade' = one month of overnight premium.")
print("See discussion_points.md §1.2 — vol_targeting is less meaningful at monthly frequency.")

## 3. Sizing Comparison

Six variants:
- **simple_bet** at 85% and 100%
- **vol_targeting** at 10% and 15% (monthly lookback = 60 months ≈ 5yr burn-in)
- **risk_based** at 1% and 2% (no stop_price → fallback sizing)

In [ ]:
# ── Simple bet ──
r_simple_85  = simple_bet(trades, bet_size=0.85)
r_simple_100 = simple_bet(trades, bet_size=1.00)

# ── Vol targeting ──
# lookback=60 months ~ 5yr. For a daily strategy this would be 60 trading days.
# At monthly frequency, we accept the longer burn-in.
r_vol_10 = vol_targeting(trades, target_vol=0.10, lookback=60)
r_vol_15 = vol_targeting(trades, target_vol=0.15, lookback=60)

# ── Risk-based (fallback: no stop_price, uses entry_price × risk_pct) ──
r_risk_1pct = risk_based(trades, risk_pct=0.01, leverage=1)
r_risk_2pct = risk_based(trades, risk_pct=0.02, leverage=1)

all_results = [r_simple_85, r_simple_100, r_vol_10, r_vol_15, r_risk_1pct, r_risk_2pct]

print("ALL IMPLEMENTATIONS:")
compare_implementations(all_results)

## 4. Equity Curves

In [ ]:
dates = [trades["entry_time"].iloc[0]] + trades["exit_time"].tolist()

fig, ax = plt.subplots(figsize=(16, 6))
for r in all_results:
    eq = r["equity_curve"]
    ret = (eq[-1] / STARTING_CAPITAL - 1) * 100
    ax.plot(dates[:len(eq)], eq, linewidth=2, alpha=0.8,
            label=f"{r['label']} — {ret:,.0f}%")
ax.axhline(STARTING_CAPITAL, color="gray", linestyle="--", alpha=0.3)
ax.set_title(f"{STRATEGY_NAME} — Sizing Methods Comparison", fontsize=14)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_ylabel("Equity ($)")
plt.tight_layout(); plt.show()

## 5. Save Implementation Stats

In [ ]:
def _key(label):
    return (label.lower()
            .replace(" ", "_").replace("(", "").replace(")", "")
            .replace("%", "pct").replace("=", "").replace(",", "")
            .replace("×", "x").replace(".", "p"))

os.makedirs("results", exist_ok=True)
equity_dir = os.path.join("results", f"{SAVE_NAME}_daily_equity")
os.makedirs(equity_dir, exist_ok=True)

impl_summary = {}
for r in all_results:
    k = _key(r["label"])
    impl_summary[k] = dict(r["stats"])
    impl_summary[k]["label"] = r["label"]

    # Save per-variant daily equity curve
    dates_full = [trades["entry_time"].iloc[0]] + trades["exit_time"].tolist()
    eq = r["equity_curve"]
    eq_series = pd.Series(eq, index=dates_full[:len(eq)])
    eq_df = eq_series.reset_index()
    eq_df.columns = ["date", "equity"]
    eq_df.to_csv(os.path.join(equity_dir, f"{k}.csv"), index=False)

# Recommended: best Sharpe with MaxDD > -50%
viable = [r for r in all_results if r["stats"]["max_dd"] > -50]
if viable:
    best = max(viable, key=lambda r: r["stats"]["sharpe"])
    impl_summary["_recommended"] = best["label"]
    print(f"Recommended: {best['label']}")
    print(f"  Sharpe={best['stats']['sharpe']}, Return={best['stats']['total_return']}%, MaxDD={best['stats']['max_dd']}%")

with open(f"results/{SAVE_NAME}_implementations.json", "w") as f:
    json.dump(impl_summary, f, indent=2)
print(f"\nSaved {len(impl_summary)-1} implementations → results/{SAVE_NAME}_implementations.json")
print(f"Per-variant equity CSVs → {equity_dir}/")